# ToolNode loop

In [7]:
import sys

# See README.md - only valid for devcontainer workflow!
sys.path.insert(0, "/workspace")


from src.tracing.phoenix import enable_langgraph_tracing

enable_langgraph_tracing(batch=True, verbose=False)

import os

endpoint = os.getenv("PHOENIX_COLLECTOR_ENDPOINT")
if not endpoint:
    raise RuntimeError(
        "Missing PHOENIX_COLLECTOR_ENDPOINT. "
        "Set it in container/compose/.env and restart the dev container."
    )
print(endpoint)


from src.reducer.base_reader import BaseReducerReader

from langgraph.graph import END, START, StateGraph
from langchain_core.messages import HumanMessage
import logging


from IPython.display import Image, display
from src.llm_nodes.global_state import GlobalState
from src.logging_setup import get_logger
from src.reducer.reducer_session import reducer_session
from src.llm_nodes.pii_email.nodes import get_pii_email_node
from src.llm_nodes.todo_extract.graph import build_todo_extract_subgraph, make_todo_extract_subgraph_runner
from src.llm_nodes.tool_node_loop.graph import (
    build_tool_node_loop_subgraph,
    make_tool_node_loop_subgraph_runner,
)
from src.llm_nodes.pii_email.mask import demask_pii_emails

MODEL = "ollama_chat/llama3.2:3b"
# MODEL = "groq/llama-3.3-70b-versatile"

pii_email = """
This is a confidential email.

Task ulf.wendel@ phpdoc dot de to buy a cup of coffee, ulf.wendel@phpdoc.de needs to plant a tree by tomorrow,
and cto@ourcompany.com needs to reach out to all employees 
regarding the current coffee shortage as it is demotivating our technical staff.
Due date, if not said otherwise, is today.
"""

GRAPH_RECURSION_LIMIT = 50

logger = get_logger(__name__, "assorted/session6/tool_node_basics.ipynb")
logger.setLevel(logging.DEBUG)


todo_extract_graph = build_todo_extract_subgraph(MODEL)
run_todo_extract_graph = make_todo_extract_subgraph_runner(todo_extract_graph)

tool_node_loop_graph = build_tool_node_loop_subgraph(MODEL)
run_tool_node_loop_graph = make_tool_node_loop_subgraph_runner(tool_node_loop_graph)

https://caddy:6006/v1/traces


In [8]:
build_graph = StateGraph(GlobalState)

build_graph.add_node("pii_extract_node", get_pii_email_node(model=MODEL))
build_graph.add_node("todo_list_node", run_todo_extract_graph)
build_graph.add_node("tool_node_loop_node", run_tool_node_loop_graph)

build_graph.add_edge(START, "pii_extract_node")
build_graph.add_edge("pii_extract_node", "todo_list_node")
build_graph.add_edge("todo_list_node", "tool_node_loop_node")
build_graph.add_edge("tool_node_loop_node", END)

graph = build_graph.compile()

# display(Image(graph.get_graph().draw_mermaid_png()))
# display(Image(tool_node_loop_graph.get_graph().draw_mermaid_png()))


In [9]:
def make_reader(get_thread_id):
    return BaseReducerReader(get_thread_id=get_thread_id)


with reducer_session("Chat-ABC", factory=make_reader) as session:
    state = session.state(GlobalState, [HumanMessage(content=pii_email)])
    reply = await session.ainvoke(
        graph,
        state,
        config={"recursion_limit": GRAPH_RECURSION_LIMIT},
    )
    state = GlobalState.model_validate(reply)

    print("TODO list:")
    for k, item in enumerate(state.todo_list.items):
        print(f"  {k + 1}): who={item.who}, what={item.what}, when={item.when}")

    print("\ntool_node_loop result (todo_text):")
    print(state.todo_text)

    print("\n--- tool loop messages ---")
    for i, m in enumerate(state.messages):
        if hasattr(m, "tool_calls") and m.tool_calls:
            print(f"[{i}] {type(m).__name__} tool_calls={m.tool_calls}")
        elif hasattr(m, "content") and isinstance(m.content, str) and m.content.strip():
            print(f"[{i}] {type(m).__name__}: {m.content[:200]}...")

    print("\nDemasked (trusted, after graph):")
    print(demask_pii_emails(state.todo_text, state.pii_email))

[DEBUG] [reducer/base_reader.py:51] observing message content: 
This is a confidential email.

Task ulf.wendel@ phpdoc dot de to buy a cup of coffee, ulf.wendel@phpdoc.de needs to plant a tree by tomorrow,
and cto@ourcompany.com needs to reach out to all employees 
regarding the current coffee shortage as it is demotivating our technical staff.
Due date, if not said otherwise, is today.

[DEBUG] [reducer/base_reader.py:51] observing message content: {"occurrences": [{"span": "ulf.wendel@ phpdoc dot de", "raw": "ulf.wendel@phpdoc.de"}, {"span": "ulf.wendel@phpdoc.de", "raw": "ulf.wendel@phpdoc.de"}, {"span": "cto@ourcompany.com", "raw": "cto@ourcompany.com"}]}
[DEBUG] [reducer/base_reader.py:51] observing message content: {"items": [{"who": "E0_03a4dc2d", "what": "plant a tree"}, {"who": "E1_03a4dc2d", "what": "reach out to all employees regarding the current coffee shortage"}]}
[DEBUG] [reducer/base_reader.py:51] observing message content: {"items": [{"who": "E0_03a4dc2d", "what": "pla

ValueError: todo_text missing placeholder who token(s): E1_03a4dc2d